# MC-C02-01: Simple Cricket Match State Microcase



## 1. Data Loading & Inspection


In [124]:
# Core imports and Colab/Drive setup.

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import pyarrow
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])
    import pyarrow

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# CHANGE ONLY THIS PATH if your Drive folder uses a different name/location.
if IN_COLAB:
  PROJECT_ROOT = Path(
    "/content/drive/My Drive/02 Work & Career/00 Current Work/"
    "Independent Sports Analytics/00 Active Projects/"
    "Analytical Casework Lab/01 — Learning Practice & Orientations/"
    "MC-C02-01 — Simple Cricket Match-State Microcase")
else:
    PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "data"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [125]:
innings_raw = RAW_DIR / "innings.csv"
matches_raw = RAW_DIR / "matches.csv"
deliveries_raw = RAW_DIR / "deliveries.csv"

In [126]:
innings_df_or = pd.read_csv(innings_raw)
matches_df_or= pd.read_csv(matches_raw)
deliveries_df_or = pd.read_csv(deliveries_raw)

print(f"Innings shape: {innings_df_or.shape}")
print(f"Matches shape: {matches_df_or.shape}")
print(f"Deliveries shape: {deliveries_df_or.shape}")

Innings shape: (80, 5)
Matches shape: (40, 8)
Deliveries shape: (9410, 11)


In [127]:
#how many non-null target values does innings.csv have, and on which innings_no
innings_df_or[innings_df_or['target'].notna()].shape

(40, 5)

In [128]:
#Check result_type in matches.csv. Does it give a count of non-completed matches, or only the missing winner?
matches_df_or['result_type'].value_counts()

,count
result_type,
completed,39
no_result,1


Preliminary Investigations

*   40 matches X 2 innings/match = 80 total innings
*   Each match contains a maximum of 2 innings
*   Maximum of 40 targets, one target for each match(none of the targets is a null value
*   Maximum number of overs in a match = 20

Data Issues
*   Delivery rows with delivery ids D000201, D000901, D001301 were duplicated - RESOLUTION: REMOVE DUPLICATED
* Match (M017) doesn't have a result and winner - RESOLUTION: Construct winner from the `deliveries` dataset - On further investigation M017 was not completed (ended at Innnings 2, Over 7) and will be excluded from further analysis







In [129]:
deliveries_df_or.duplicated().sum()

np.int64(3)

In [130]:
#identify the duplicated rows
deliveries_df_or[deliveries_df_or.duplicated(keep=False)].sort_values(by=['match_id','innings_no','over_no'])

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
200,D000201,M001,2,11,6,B05,P201,0,0,NaN,0
9407,D000201,M001,2,11,6,B05,P201,0,0,NaN,0
900,D000901,M004,2,7,1,B01,P201,6,0,NaN,0
9408,D000901,M004,2,7,1,B01,P201,6,0,NaN,0
1300,D001301,M006,1,11,6,B05,P109,1,0,NaN,0
9409,D001301,M006,1,11,6,B05,P109,1,0,NaN,0


In [131]:
deliveries_df_or.delivery_id.duplicated(keep=False).sum()

np.int64(6)

In [132]:
#idenitify rows with duplicated delivery ids
deliveries_df_or[deliveries_df_or.delivery_id.duplicated(keep=False)].sort_values(by=['match_id','innings_no','over_no'])

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
200,D000201,M001,2,11,6,B05,P201,0,0,NaN,0
9407,D000201,M001,2,11,6,B05,P201,0,0,NaN,0
900,D000901,M004,2,7,1,B01,P201,6,0,NaN,0
9408,D000901,M004,2,7,1,B01,P201,6,0,NaN,0
1300,D001301,M006,1,11,6,B05,P109,1,0,NaN,0
9409,D001301,M006,1,11,6,B05,P109,1,0,NaN,0


In [133]:
#make copies of the raw dfs and exclude M017 from the copies and the duplicates from the deliveries_df
innings_df = innings_df_or.copy()
matches_df = matches_df_or.copy()
deliveries_df = deliveries_df_or.copy()

innings_df = innings_df[innings_df['match_id'] != 'M017']
matches_df = matches_df[matches_df['match_id'] != 'M017']
deliveries_df = deliveries_df[deliveries_df['match_id'] != 'M017']

deliveries_df = deliveries_df[~deliveries_df.delivery_id.duplicated()]

In [134]:
len(deliveries_df)

9241

In [135]:
print("--- Innings Head ---")
display(innings_df.head())
print("\n--- Matches Head ---")
display(matches_df.head())
print("\n--- Deliveries Head ---")
display(deliveries_df.head())

--- Innings Head ---


,match_id,innings_no,batting_team,bowling_team,target
0,M001,1,Jinja Jets,Entebbe Eagles,NaN
1,M001,2,Entebbe Eagles,Jinja Jets,215.0
2,M002,1,Mbale Mambas,Kampala Kings,NaN
3,M002,2,Kampala Kings,Mbale Mambas,195.0
4,M003,1,Entebbe Eagles,Gulu Giants,NaN



--- Matches Head ---


,match_id,match_date,venue,team_a,team_b,toss_winner,result_type,winner
0,M001,2026-03-04,Jinja Park,Jinja Jets,Entebbe Eagles,Jinja Jets,completed,Jinja Jets
1,M002,2026-03-07,Lugogo Oval,Kampala Kings,Mbale Mambas,Mbale Mambas,completed,Mbale Mambas
2,M003,2026-03-10,Jinja Park,Gulu Giants,Entebbe Eagles,Entebbe Eagles,completed,Entebbe Eagles
3,M004,2026-03-13,Lugogo Oval,Gulu Giants,Entebbe Eagles,Entebbe Eagles,completed,Gulu Giants
4,M005,2026-03-16,Entebbe Ground,Mbale Mambas,Gulu Giants,Mbale Mambas,completed,Mbale Mambas



--- Deliveries Head ---


,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
0,D000001,M001,1,1,1,B01,P101,4,0,NaN,0
1,D000002,M001,1,1,2,B01,P101,0,1,legbye,0
2,D000003,M001,1,1,3,B01,P101,1,0,NaN,0
3,D000004,M001,1,1,4,B01,P101,4,1,noball,0
4,D000005,M001,1,1,5,B01,P101,1,0,NaN,0


In [136]:
print(f"Innings shape: {innings_df.shape}")
print(f"Matches shape: {matches_df.shape}")
print(f"Deliveries shape: {deliveries_df.shape}")

Innings shape: (78, 5)
Matches shape: (39, 8)
Deliveries shape: (9241, 11)


## 2. Row grain, keys and data Issues


In [137]:
deliveries_df.head(2)

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
0,D000001,M001,1,1,1,B01,P101,4,0,NaN,0
1,D000002,M001,1,1,2,B01,P101,0,1,legbye,0


- One row in the `deliveries` dataset represents one legal or illegal ball delivery event and its outcome, a wide is an illegal delivery


In [138]:
deliveries_df[deliveries_df['extra_type'] == 'wide'].head()

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
6,D000007,M001,1,1,7,B01,P101,0,1,wide,0
10,D000011,M001,1,2,3,B02,P102,0,1,wide,0
27,D000028,M001,1,5,1,B05,P104,0,1,wide,0
41,D000042,M001,1,7,2,B01,P105,0,1,wide,0
61,D000062,M001,1,10,3,B04,P107,0,1,wide,0


In [139]:
deliveries_df[deliveries_df['extras'] == 1].shape

(619, 11)

In [140]:
deliveries_df['extra_type'].isna().sum()

np.int64(8622)

In [141]:
deliveries_df['extra_type'].unique()

array([nan, 'legbye', 'noball', 'wide'], dtype=object)

In [142]:
deliveries_df['extra_type'].value_counts()

,count
extra_type,
wide,326
legbye,185
noball,108


 `delivery_id` is unique and identifies one delivery event because the number of unique values is the same as the length of the dataset and there arent any duplicated values

In [143]:
deliveries_df['delivery_id'].nunique()

9241

In [144]:
deliveries_df['delivery_id'].duplicated().sum()

np.int64(0)

In [145]:
innings_df.head(5)

,match_id,innings_no,batting_team,bowling_team,target
0,M001,1,Jinja Jets,Entebbe Eagles,NaN
1,M001,2,Entebbe Eagles,Jinja Jets,215.0
2,M002,1,Mbale Mambas,Kampala Kings,NaN
3,M002,2,Kampala Kings,Mbale Mambas,195.0
4,M003,1,Entebbe Eagles,Gulu Giants,NaN


- One row in the `innings_df` represents one innings of a match
- The keys that uniquely identify a row in the dataset are the `match_id` and `innings_no` columns together


In [146]:
matches_df.head(5)

,match_id,match_date,venue,team_a,team_b,toss_winner,result_type,winner
0,M001,2026-03-04,Jinja Park,Jinja Jets,Entebbe Eagles,Jinja Jets,completed,Jinja Jets
1,M002,2026-03-07,Lugogo Oval,Kampala Kings,Mbale Mambas,Mbale Mambas,completed,Mbale Mambas
2,M003,2026-03-10,Jinja Park,Gulu Giants,Entebbe Eagles,Entebbe Eagles,completed,Entebbe Eagles
3,M004,2026-03-13,Lugogo Oval,Gulu Giants,Entebbe Eagles,Entebbe Eagles,completed,Gulu Giants
4,M005,2026-03-16,Entebbe Ground,Mbale Mambas,Gulu Giants,Mbale Mambas,completed,Mbale Mambas


- One row in the `matches_df` represents a single match that was played between two teams and its outcome
- The key that uniquely identifies a row in the dataset is the `match_id` column; When the deliveries df is joined to innings df on match_id alone the rows are doubled

In [147]:
pd.merge(deliveries_df,innings_df,on=['match_id','innings_no']).head()

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket,batting_team,bowling_team,target
0,D000001,M001,1,1,1,B01,P101,4,0,NaN,0,Jinja Jets,Entebbe Eagles,NaN
1,D000002,M001,1,1,2,B01,P101,0,1,legbye,0,Jinja Jets,Entebbe Eagles,NaN
2,D000003,M001,1,1,3,B01,P101,1,0,NaN,0,Jinja Jets,Entebbe Eagles,NaN
3,D000004,M001,1,1,4,B01,P101,4,1,noball,0,Jinja Jets,Entebbe Eagles,NaN
4,D000005,M001,1,1,5,B01,P101,1,0,NaN,0,Jinja Jets,Entebbe Eagles,NaN


## 3. Definitions

In [148]:
deliveries_df['extra_type'].value_counts()

,count
extra_type,
wide,326
legbye,185
noball,108


In [149]:
deliveries_df['extra_type'].isna().sum()

np.int64(8622)

In [150]:
def get_surrounding_deliveries(df, delivery_id, window=3):
    # Reset index to ensure we have standard integer positioning
    df_temp = df.reset_index(drop=True)

    # Find the index of the matching delivery_id
    matches = df_temp[df_temp['delivery_id'] == delivery_id].index

    if len(matches) == 0:
        print(f"Delivery ID '{delivery_id}' not found.")
        return None

    idx = matches[0]

    # Calculate start and end bounds
    start_idx = max(0, idx - window)
    end_idx = min(len(df_temp), idx + window + 1)

    return df_temp.iloc[start_idx:end_idx]

# Example usage for delivery 'D000071'
surrounding_df = get_surrounding_deliveries(deliveries_df, 'D000034', window=3)
display(surrounding_df)

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
30,D000031,M001,1,5,4,B05,P104,4,0,NaN,0
31,D000032,M001,1,5,5,B05,P104,4,0,NaN,0
32,D000033,M001,1,5,6,B05,P104,2,0,NaN,0
33,D000034,M001,1,5,7,B05,P104,0,0,NaN,1
34,D000035,M001,1,6,1,B06,P105,0,0,NaN,0
35,D000036,M001,1,6,2,B06,P105,1,0,NaN,0
36,D000037,M001,1,6,3,B06,P105,0,0,NaN,0


In [151]:
deliveries_df[(deliveries_df['match_id'] == 'M001') & (deliveries_df['innings_no'] == 1) & (deliveries_df['over_no'] == 11)]

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket
66,D000067,M001,1,11,1,B05,P107,6,0,NaN,0
67,D000068,M001,1,11,2,B05,P107,1,1,noball,0
68,D000069,M001,1,11,3,B05,P107,1,0,NaN,0
69,D000070,M001,1,11,4,B05,P107,6,0,NaN,0
70,D000071,M001,1,11,5,B05,P107,0,1,wide,0
71,D000072,M001,1,11,6,B05,P107,0,1,legbye,0
72,D000073,M001,1,11,7,B05,P107,0,0,NaN,0
73,D000074,M001,1,11,8,B05,P107,2,0,NaN,0


In [152]:
# A legal ball is whose extra_type is either NaN or a legbye, all other extra_types are not legal balls
deliveries_df['is_legal_ball'] = deliveries_df['extra_type'].isna() | (deliveries_df['extra_type'] == 'legbye')

In [153]:
deliveries_df['wicket'].value_counts()

,count
wicket,
0,8765
1,476


- Sum of `runs_off_bat` and `extras` before or at the checkpoint count towards a team total at the checkpoint
- Sum of `wicket` before or at the checkpoint count towards the total number of wickets




In [154]:
deliveries_df['over_no'].describe()

,over_no
count,9241.000000
mean,10.082675
std,5.626880
min,1.000000
25%,5.000000
50%,10.000000
75%,15.000000
max,20.000000


The checkpoint is `over_no <= 10` because it includes extras earned even when the deliveries were not legal balls

### Investigating Innings Ending Before 10 Overs

Let's find all `(match_id, innings_no)` combinations where the maximum `over_no` in the deliveries dataset is 10 or fewer.

In [155]:
# Calculate the max over recorded for each match and innings
innings_max_overs = deliveries_df.groupby(['match_id', 'innings_no'])['over_no'].max().reset_index()

# Filter for innings that completed with 10 or fewer overs
short_innings = innings_max_overs[innings_max_overs['over_no'] <= 10]

print(f"Found {len(short_innings)} innings that ended at or before 10 overs:")
display(short_innings)

if len(short_innings) > 0:
    # Let's inspect the details of one such short innings if any exist
    sample_match = short_innings.iloc[0]['match_id']
    sample_innings = short_innings.iloc[0]['innings_no']
    print(f"\nPreviewing final deliveries of Match {sample_match}, Innings {sample_innings}:")
    display(deliveries_df[(deliveries_df['match_id'] == sample_match) & (deliveries_df['innings_no'] == sample_innings)].tail(5))


Found 2 innings that ended at or before 10 overs:


,match_id,innings_no,over_no
58,M031,1,9
59,M031,2,6



Previewing final deliveries of Match M031, Innings 1:


,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket,is_legal_ball
7227,D007228,M031,1,9,2,B03,P109,0,0,NaN,0,True
7228,D007229,M031,1,9,3,B03,P109,0,0,NaN,1,True
7229,D007230,M031,1,9,4,B03,P110,0,0,NaN,0,True
7230,D007231,M031,1,9,5,B03,P110,0,0,NaN,0,True
7231,D007232,M031,1,9,6,B03,P110,2,0,NaN,1,True


### Verifying Wicket Counts for Short Innings in Match M031

In [156]:
# Calculate the total wickets taken in both innings of match M031
m031_wickets = deliveries_df[deliveries_df['match_id'] == 'M031'].groupby('innings_no')['wicket'].sum().reset_index()
print("Total wickets recorded for each innings in Match M031:")
display(m031_wickets)

Total wickets recorded for each innings in Match M031:


,innings_no,wicket
0,1,10
1,2,4


Found two innings **that** ended before 10 oversMatch M031,innings 1 with 9 overs ended because the batting team was all our and Match M031 innings 2 with 6 overs, because they chased down the taret score and won the match early.



In [157]:
#the analysis is going to comprise only chases that reached the checkpoint, so we are going to remove Match 031 from the datasets
deliveries_df = deliveries_df[deliveries_df['match_id'] != 'M031']
innings_df = innings_df[innings_df['match_id'] != 'M031']
matches_df = matches_df[matches_df['match_id'] != 'M031']

In [158]:
#test whether match_id + innings_no + over_no + ball_seq_in_over is also unique, as a natural-key cross-check.
deliveries_df['key'] = deliveries_df['match_id'] + deliveries_df['innings_no'].astype(str) + deliveries_df['over_no'].astype(str) + deliveries_df['ball_seq_in_over'].astype(str)
deliveries_df['key'].duplicated().sum()

np.int64(0)

 This confirms that the combination of match_id + innings_no + over_no + ball_seq_in_over acts as a perfectly unique natural key for each delivery in your dataset.

In [159]:
del deliveries_df['key']

## 4. Part A: M002 warm-up (pandas)

In [160]:
M002_innings2_deliveries_df = deliveries_df[(deliveries_df['match_id'] == 'M002') & (deliveries_df['innings_no'] == 2)]
M002_innings2_deliveries_df.head()

,delivery_id,match_id,innings_no,over_no,ball_seq_in_over,bowler_id,batter_id,runs_off_bat,extras,extra_type,wicket,is_legal_ball
358,D000359,M002,2,1,1,B01,P201,2,0,NaN,0,True
359,D000360,M002,2,1,2,B01,P201,0,0,NaN,0,True
360,D000361,M002,2,1,3,B01,P201,1,0,NaN,0,True
361,D000362,M002,2,1,4,B01,P201,4,0,NaN,0,True
362,D000363,M002,2,1,5,B01,P201,1,1,noball,0,False


In [161]:
M002_innings2_runs = M002_innings2_deliveries_df['runs_off_bat'].sum() + M002_innings2_deliveries_df['extras'].sum()
M002_innings2_wickets = M002_innings2_deliveries_df['wicket'].sum()
M002_innings2_legal_balls = M002_innings2_deliveries_df['is_legal_ball'].sum()

print(f"M002 Innings 2 Runs: {M002_innings2_runs}")
print(f"M002 Innings 2 Wickets: {M002_innings2_wickets}")
print(f"M002 Innings 2 Legal Balls: {M002_innings2_legal_balls}")

M002 Innings 2 Runs: 171
M002 Innings 2 Wickets: 6
M002 Innings 2 Legal Balls: 120


In [162]:
#create df for the first 10 overs in the second innings
M002_innings2_10_overs_df = M002_innings2_deliveries_df[M002_innings2_deliveries_df['over_no'] <= 10]

In [163]:
M002_innings2_10_overs_runs = M002_innings2_10_overs_df['runs_off_bat'].sum() + M002_innings2_10_overs_df['extras'].sum()
M002_innings2_10_overs_wickets = M002_innings2_10_overs_df['wicket'].sum()
M002_innings2_10_overs_legal_balls = M002_innings2_10_overs_df['is_legal_ball'].sum()

print(f"M002 Innings 2 Runs in first 10 overs: {M002_innings2_10_overs_runs}")
print(f"M002 Innings 2 Wickets in first 10 overs: {M002_innings2_10_overs_wickets}")
print(f"M002 Innings 2 Legal Balls in first 10 overs: {M002_innings2_10_overs_legal_balls}")

M002 Innings 2 Runs in first 10 overs: 83
M002 Innings 2 Wickets in first 10 overs: 5
M002 Innings 2 Legal Balls in first 10 overs: 60


## 5. Part A: M002 warm-up (SQL) and reconciliation


In [164]:
import sqlite3

SQL_DIR = PROJECT_ROOT / "sql"
SQL_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = SQL_DIR / "mc_c02_01_cricket_match_state_analysis.sqlite"

# ---------------------------------------------------------
# 1. Prepare relational tables for M002
# ---------------------------------------------------------

sql_deliveries_df = deliveries_df_or.copy()

# ---------------------------------------------------------
# 2. Write table to SQLite
# ---------------------------------------------------------
with sqlite3.connect(DB_PATH) as conn:
    sql_deliveries_df.to_sql('sql_deliveries_df', conn, if_exists='replace', index=False)


In [165]:
query = """
SELECT
    SUM(runs_off_bat) + SUM(extras) AS total_runs,
    SUM(wicket) AS total_wickets,
    SUM(CASE WHEN (extra_type IS NULL OR extra_type = 'legbye') THEN 1 ELSE 0 END) AS total_legal_balls
FROM sql_deliveries_df
WHERE
  match_id = 'M002'
AND
  innings_no = 2
AND
  over_no <= 10;
"""

with sqlite3.connect(DB_PATH) as conn:
    query_output = pd.read_sql_query(
        query,
        conn
    )

display(query_output)

,total_runs,total_wickets,total_legal_balls
0,83,5,60


## 6. Part A: rows versus events diagnostic

The SQL and pandas implementation results both reconcile on the total runs, wickets and legal balls

In [166]:
# Compare the 10-overs pandas variables with the 10-overs SQL query outputs
print(f"Pandas Runs: {M002_innings2_10_overs_runs} | SQL Runs: {query_output['total_runs'][0]}")
print(f"Pandas Wickets: {M002_innings2_10_overs_wickets} | SQL Wickets: {query_output['total_wickets'][0]}")
print(f"Pandas Legal Balls: {M002_innings2_10_overs_legal_balls} | SQL Legal Balls: {query_output['total_legal_balls'][0]}")

assert M002_innings2_10_overs_runs == query_output['total_runs'][0]
assert M002_innings2_10_overs_wickets == query_output['total_wickets'][0]
assert M002_innings2_10_overs_legal_balls == query_output['total_legal_balls'][0]
print("Reconciliation Successful!")

Pandas Runs: 83 | SQL Runs: 83
Pandas Wickets: 5 | SQL Wickets: 5
Pandas Legal Balls: 60 | SQL Legal Balls: 60
Reconciliation Successful!


In [167]:
len(M002_innings2_10_overs_df)

65

In [168]:
M002_innings2_10_overs_df['is_legal_ball'].sum()

np.int64(60)

In [169]:
M002_innings2_10_overs_df[['extra_type']].value_counts()

,count
extra_type,
noball,3
wide,2
legbye,1


65 rows in MD002, with 60 legal balls and 5 extras(3 noballs and 2 wides)

In [170]:
matches_df['match_id'].nunique()

38

In [171]:
matches_df.columns

Index(['match_id', 'match_date', 'venue', 'team_a', 'team_b', 'toss_winner',
       'result_type', 'winner'],
      dtype='object')

## 7. Plan for all chases

- Unit of analysis: one row(a chase) is going to represent the match state at the end of the first 10 overs in the second innings of each valid match. This match state will be defined by a number of columns. The key will be the `Match_ID` since each match has a single chase

- Columns the table needs include the `match_ID`,`total_runs`,`wickets`,`target`,`legal_balls`,`CRR`,`RRR`,`gap`,`bucket`,`outcome`. `total_runs` uses the stage 3 rule of `runs_off_bat` + `extras`, `legal_balls` still uses the stage 3 rule of legal balls either being null or legbyes . `target` column comes from the `innings` table joned on the `match_ID` column. `outcome` column will be got from summiing the total runs from the second innings of each match and comparing them witht the `target`. if the total runs in the seond innings are greater than or equal to the target, then `outcome = 1`, else `outcome=0`

- The checkpoint is `over_no <= 10` . If a chase ends in the middle of the 10th over `sum of is_legal_ball < 60`, then we exclude it, because it skews the CRR value which strictly divides by 10 overs. If the wickets were all out before the end of the chase ie `sum of wickets = 10`, we also exclude them , if the target is reached at the last legal ball of the chase it reaches the checkpoint, because it meets the 60 legal balls and 10 overs which are needed for the division.

- Numerator and denominator of each bucket's chase-win rate: numerator is total wins `sum of outcome` in the bucket, denominator is the total number of chases in the bucket (Total number of rows in bucket)

- Validation of the counts (expected row count- count total number of rows in df which should be equal to *38* the number of matches with complete/valid overs, bucket counts should add up to the total of all chases in the df, exclusions accounted for), bucket counts sum should be equal to 38, there shoulnt be any null values in `gap`, `bucket` or `outcome` columns.


In [172]:
deliveries_df['match_id'].nunique()

38

In [173]:
deliveries_df.columns

Index(['delivery_id', 'match_id', 'innings_no', 'over_no', 'ball_seq_in_over',
       'bowler_id', 'batter_id', 'runs_off_bat', 'extras', 'extra_type',
       'wicket', 'is_legal_ball'],
      dtype='object')

In [ ]:
#Generate the final chase table
#start by filtering the deliveries to only capture the first 10 overs in the second innings of each match
deliveries_df_10_overs = deliveries_df[(deliveries_df['over_no'] <= 10) & (deliveries_df['innings_no'] == 2)]

#aggregating the required columns(total_runs_off_ball,extras,wickets,legal_balls_bowled) grouped by Match_ID
deliveries_df_10_overs_agg = deliveries_df_10_overs.groupby('match_id').agg(
    total_runs_off_bat = ('runs_off_bat','sum'),
    total_extras = ('extras','sum'),
    total_wickets = ('wicket','sum'),
    total_legal_balls = ('is_legal_ball','sum')
)

#remove the rows that do not meet the checkpoint criteria(total_legal_balls < 60 or total_wickets = 10)
deliveries_df_10_overs_agg.drop(deliveries_df_10_overs_agg[(deliveries_df_10_overs_agg['total_legal_balls'] < 60) | (deliveries_df_10_overs_agg['total_wickets'] == 10)].index, inplace=True)

#create the 'total_runs' column
deliveries_df_10_overs_agg['total_runs'] = deliveries_df_10_overs_agg['total_runs_off_bat'] + deliveries_df_10_overs_agg['total_extras']



,total_runs_off_bat,total_extras,total_wickets,total_legal_balls,total_runs
match_id,,,,,
M001,95,8,0,60,103
M002,77,6,5,60,83
M003,64,2,2,60,66
M004,106,5,0,60,111
M005,86,5,2,60,91


In [175]:
#create a copy of the aggregated deliveries table to create the final chase table
chase_df = deliveries_df_10_overs_agg.copy().reset_index()
chase_df.drop(columns=['total_runs_off_bat','total_extras'],inplace=True)

#select only rows from the second innings
second_innings = innings_df[innings_df['innings_no'] == 2][['match_id','target']]

#merge the target column from the innings table to the chase table,
chase_df = pd.merge(chase_df, second_innings, on = 'match_id')
chase_df['target'] = chase_df['target'].astype('int')

#adding CRR(current run rate)
chase_df['CRR'] = chase_df['total_runs'] / 10

#RRR(required run rate) = (target - runs scored) / (balls remaining / 6), balls remaining = 120 - legal balls bowled at the checkpoint
chase_df['balls_remaining'] = 120 - chase_df['total_legal_balls']
chase_df['RRR'] = (chase_df['target'] - chase_df['total_runs']) / (chase_df['balls_remaining'] / 6)

#gap = RRR - CRR
chase_df['gap'] = chase_df['RRR'] - chase_df['CRR']


In [176]:
# creating Buckets for the chase df: gap <= 0 | 0 < gap <= 2 | gap > 2
chase_df['gap_bucket'] = pd.cut(chase_df['gap'], bins=[-np.inf, 0, 2, np.inf], labels=['gap <= 0', '0 < gap <= 2', 'gap > 2'])

In [ ]:
# Total number of runs scored in the second innings per match
second_innings_runs = deliveries_df[deliveries_df['innings_no'] == 2].groupby('match_id').agg(
    total_runs_off_bat=('runs_off_bat', 'sum'),
    total_extras=('extras', 'sum')
).reset_index()

second_innings_runs['total_second_innings_runs'] = second_innings_runs['total_runs_off_bat'] + second_innings_runs['total_extras']


,match_id,total_runs_off_bat,total_extras,total_second_innings_runs
0,M001,190,10,200
1,M002,161,10,171
2,M003,158,7,165
3,M004,179,9,188
4,M005,167,13,180


In [178]:
#creating the outcome column in the chase_df from the second_innings_run
chase_df = pd.merge(chase_df, second_innings_runs[['match_id', 'total_second_innings_runs']], on='match_id', how='left')
chase_df['outcome'] = np.where(chase_df['total_second_innings_runs'] >= chase_df['target'], 1, 0)

In [179]:
chase_df.head()

,match_id,total_wickets,total_legal_balls,total_runs,target,CRR,balls_remaining,RRR,gap,gap_bucket,total_second_innings_runs,outcome
0,M001,0,60,103,215,10.3,60,11.2,0.9,0 < gap <= 2,200,0
1,M002,5,60,83,195,8.3,60,11.2,2.9,gap > 2,171,0
2,M003,2,60,66,224,6.6,60,15.8,9.2,gap > 2,165,0
3,M004,0,60,111,183,11.1,60,7.2,-3.9,gap <= 0,188,1
4,M005,2,60,91,209,9.1,60,11.8,2.7,gap > 2,180,0


In [182]:
# Validation and sanity checks on chase_df
chase_df.info()
chase_df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   match_id                   38 non-null     object  
 1   total_wickets              38 non-null     int64   
 2   total_legal_balls          38 non-null     int64   
 3   total_runs                 38 non-null     int64   
 4   target                     38 non-null     int64   
 5   CRR                        38 non-null     float64 
 6   balls_remaining            38 non-null     int64   
 7   RRR                        38 non-null     float64 
 8   gap                        38 non-null     float64 
 9   gap_bucket                 38 non-null     category
 10  total_second_innings_runs  38 non-null     int64   
 11  outcome                    38 non-null     int64   
dtypes: category(1), float64(3), int64(7), object(1)
memory usage: 3.6+ KB


,total_wickets,total_legal_balls,total_runs,target,CRR,balls_remaining,RRR,gap,total_second_innings_runs,outcome
count,38.000000,38.0,38.000000,38.000000,38.000000,38.0,38.000000,38.000000,38.000000,38.000000
mean,2.789474,60.0,90.500000,188.289474,9.050000,60.0,9.778947,0.728947,172.894737,0.447368
std,1.773040,0.0,14.773515,26.804505,1.477352,0.0,2.921787,3.775448,20.721825,0.503897
min,0.000000,60.0,66.000000,129.000000,6.600000,60.0,3.600000,-7.700000,133.000000,0.000000
25%,2.000000,60.0,79.000000,171.250000,7.900000,60.0,7.950000,-1.575000,157.250000,0.000000
50%,2.000000,60.0,90.000000,191.000000,9.000000,60.0,9.800000,0.950000,174.500000,0.000000
75%,4.000000,60.0,98.000000,209.750000,9.800000,60.0,11.200000,2.650000,187.500000,1.000000
max,7.000000,60.0,129.000000,234.000000,12.900000,60.0,15.800000,9.200000,223.000000,1.000000


In [183]:
chase_df['gap_bucket'].value_counts()

,count
gap_bucket,
gap <= 0,14
gap > 2,13
0 < gap <= 2,11


In [184]:
chase_df['outcome'].value_counts()

,count
outcome,
0,21
1,17
